In [3]:
# lectura de los datos

import pandas as pd
import read_data

# Suponiendo que el activo tiene ese nombre
# solo es remplazar por el nombre que se tenga
# solo se acepta archivos .parquet y .csv
nombre_activo: str = "EURUSD_2019-2025.parquet"
data: pd.DataFrame = pd.read_parquet(f"Data/{nombre_activo}")

primer_lunes = data[data.index.dayofweek == 0].index[0]
ultimo_viernes = data[data.index.dayofweek == 4].index[-1]

data = data.loc[primer_lunes: ultimo_viernes]
data = data.between_time('07:00', '16:00')
data

,Precio Spot
time,
2019-01-07 07:00:00,1.142020
2019-01-07 07:01:00,1.142060
2019-01-07 07:02:00,1.142060
2019-01-07 07:03:00,1.142180
2019-01-07 07:04:00,1.142170
...,...
2026-02-13 15:56:00,1.187810
2026-02-13 15:57:00,1.187695
2026-02-13 15:58:00,1.187485


In [5]:
import numpy as np
import pandas as pd

ma_week = []

with open("Data/Moving_average_test.txt", "r") as archivo:

    for line in archivo:
        start = pd.Timestamp(line[:10])
        fin = pd.Timestamp(line[11:21])

        lista = eval(line[22: ], {"np": np})

        ma_week.append([start, fin, lista])

In [3]:
import train_network

X, y = train_network.build_weekly_dataset(data, ma_week)

Calculando matemáticas avanzadas para timeframe 13min (solo se hace una vez)...
Calculando matemáticas avanzadas para timeframe 11min (solo se hace una vez)...
Calculando matemáticas avanzadas para timeframe 12min (solo se hace una vez)...
Calculando matemáticas avanzadas para timeframe 14min (solo se hace una vez)...
Calculando matemáticas avanzadas para timeframe 19min (solo se hace una vez)...
Calculando matemáticas avanzadas para timeframe 8min (solo se hace una vez)...
Calculando matemáticas avanzadas para timeframe 15min (solo se hace una vez)...
Calculando matemáticas avanzadas para timeframe 16min (solo se hace una vez)...
Calculando matemáticas avanzadas para timeframe 4min (solo se hace una vez)...
Calculando matemáticas avanzadas para timeframe 10min (solo se hace una vez)...
Calculando matemáticas avanzadas para timeframe 5min (solo se hace una vez)...
Calculando matemáticas avanzadas para timeframe 20min (solo se hace una vez)...
Calculando matemáticas avanzadas para timef

In [5]:
import train_network

model = train_network.train_model(X, y)

              precision    recall  f1-score   support

           0       0.66      0.60      0.63      6852
           1       0.64      0.70      0.67      7034

    accuracy                           0.65     13886
   macro avg       0.65      0.65      0.65     13886
weighted avg       0.65      0.65      0.65     13886



In [16]:
import joblib

nombre_archivo = "modelo_filtro_ma.joblib"
joblib.dump(model, nombre_archivo)

print(f"¡Modelo guardado exitosamente como {nombre_archivo}!")

¡Modelo guardado exitosamente como modelo_filtro_ma.joblib!


In [3]:
import joblib

model = joblib.load("modelo_filtro_ma.joblib")

In [26]:
from tester import backtest
from use_tecnics import main

def get_info_in_week(data: pd.DataFrame, week: list) -> pd.DataFrame:
    time_col = week[0] - pd.Timedelta(weeks=1)

    data_week_raw = data.loc[time_col: week[1]]

    velas_cerradas = read_data.ohlc_form(
        data_week_raw,
        f"{week[2][1]}min"
    )["close"]

    method_name = week[2][0]
    params = week[2][2]

    df_week = main(method_name, velas_cerradas, params)

    df_week = df_week.loc[week[0]: week[1]]

    if len(df_week) == 0:
        return pd.DataFrame()

    return df_week

def get_signals_and_prices(data: pd.DataFrame, ma_week: list) -> pd.DataFrame:
    signals_and_prices = []

    for week in ma_week:
        data_week = get_info_in_week(data, week)

        if len(data_week) == 0:
            continue

        if len(data_week) > 1:
            f_i = data_week.index[0]
            l_i = data_week.index[-1]

            if data_week["Signals"][f_i] == -1:
                f_i = data_week.index[1]

            if data_week["Signals"][l_i] == 1:
                l_i = data_week.index[-2]

            data_week = data_week.loc[f_i: l_i]
        
        signals_and_prices.append(data_week)

    if len(signals_and_prices) == 0:
        return pd.DataFrame()

    signals_and_prices = pd.concat(signals_and_prices)

    signals_and_prices = signals_and_prices.sort_index()
    signals_and_prices = signals_and_prices[~signals_and_prices.index.duplicated()]

    return signals_and_prices

signals_and_prices_c = get_signals_and_prices(data, ma_week[-5: -4])
backtest(signals_and_prices_c, True)

(np.float64(0.5714285714285714),
 np.float64(0.3148292578229086),
 np.float64(0.4197723437638782),
 7)

In [27]:
signals_and_prices_c

,Signals,Prices
time,,
2019-11-04 21:20:00,1,1.11552
2019-11-06 11:40:00,-1,1.10914
2019-11-06 12:00:00,1,1.10894
2019-11-06 14:40:00,-1,1.10896
2019-11-06 15:40:00,1,1.10879
2019-11-06 22:20:00,-1,1.10870
2019-11-07 07:00:00,1,1.10594
2019-11-07 10:40:00,-1,1.10831
2019-11-07 11:20:00,1,1.10766


In [5]:
from train_network import filter_signals_with_ml

signals_and_prices = filter_signals_with_ml(signals_and_prices_c, data, model, umbral=.6)

In [21]:
backtest(signals_and_prices)

(np.float64(0.8461538461538461),
 np.float64(0.5079688586267431),
 np.float64(2.869337606837549),
 247)

In [16]:
signals_and_prices = signals_and_prices.loc[signals_and_prices["Signals"] != 0]
signals_and_prices = signals_and_prices.loc[signals_and_prices["Signals"] != signals_and_prices["Signals"].shift(1)]
signals_and_prices

,Signals,Prices
time,,
2025-04-02 23:24:00,1,1.083615
2025-04-02 23:51:00,-1,1.086025
2025-04-03 00:09:00,1,1.082920
2025-04-03 02:06:00,-1,1.085520
2025-04-03 18:36:00,1,1.104500
...,...,...
2026-02-11 09:52:00,-1,1.191690
2026-02-11 17:06:00,1,1.185830
2026-02-11 18:16:00,-1,1.187020


In [28]:
import pandas as pd
import matplotlib.pyplot as plt

def simular_trading_basico(señales_precios, capital_inicial=1000.0, valor_pip=0.0001):
    compradas = 0
    capital_operativo = capital_inicial

    pico_maximo = capital_inicial
    max_drawdown = 0.0
    
    for idx in señales_precios.index:
        precio_actual = señales_precios["Prices"][idx]
        senal = señales_precios["Signals"][idx]

        if senal == 1:
            precio_compra = precio_actual + valor_pip
            if capital_operativo >= precio_compra:
                nuevas_compras = int(capital_operativo / precio_compra)
                compradas += nuevas_compras
                capital_operativo -= nuevas_compras * precio_compra

        elif senal == -1: 
            precio_venta = precio_actual - valor_pip
            capital_operativo += compradas * precio_venta
            compradas = 0
                
        patrimonio_actual = capital_operativo + (compradas * precio_actual)

        if patrimonio_actual > pico_maximo:
            pico_maximo = patrimonio_actual
            
        drawdown_actual = (pico_maximo - patrimonio_actual) / pico_maximo
        
        if drawdown_actual > max_drawdown:
            max_drawdown = drawdown_actual

    print(f"--- RESULTADOS DEL SIMULADOR (INTERÉS COMPUESTO) ---")
    print(f"Patrimonio Total Final: ${patrimonio_actual:.2f}")
    print(f"Capital Inicial: ${capital_inicial:.2f}")
    print(f"Beneficio Neto: ${(patrimonio_actual - capital_inicial):.2f}")
    print(f"Max Drawdown: {max_drawdown * 100:.2f}%\n")
    
    return patrimonio_actual, max_drawdown

simular_trading_basico(signals_and_prices_c)
#simular_trading_basico(signals_and_prices)
print("")

--- RESULTADOS DEL SIMULADOR (INTERÉS COMPUESTO) ---
Patrimonio Total Final: $996.42
Capital Inicial: $1000.00
Beneficio Neto: $-3.58
Max Drawdown: 0.64%




In [2]:
import pandas as pd
import find_best
import keys
from use_tecnics import simple_methods

keys.methods = simple_methods
keys.candles = 20
keys.calls = 20

dias_comp_sem = pd.Timedelta(days=4)
next_lun = pd.Timedelta(days=3)
ventana_ent = pd.Timedelta(weeks=3) + dias_comp_sem

primer_lunes = data[data.index.dayofweek == 0].index[0]
ultimo_viernes = data[data.index.dayofweek == 4].index[-1]

inicio_actual = primer_lunes

with open("Data/Moving_average_test.txt", "w") as archivo:
    archivo.write("") 

while True:
    train_time_init = inicio_actual
    train_time_fin = inicio_actual + ventana_ent

    test_time_init = train_time_fin + next_lun
    test_time_fin = test_time_init + dias_comp_sem

    if train_time_fin >= ultimo_viernes:
        break

    print(f"Optimizando ventana: {train_time_init.date()} al {train_time_fin.date()}")

    data_w = data.loc[train_time_init : train_time_fin]
    
    if len(data_w) < 100:
        inicio_actual += (dias_comp_sem + next_lun)
        continue

    best_ma = find_best.opti_main(data_w)

    with open("Data/Moving_average_test.txt", "a") as archivo:
        archivo.write(f"{test_time_init.date()} {test_time_fin.date()}, {best_ma}\n")

    inicio_actual += (dias_comp_sem + next_lun)

Optimizando ventana: 2019-01-07 al 2019-02-01

RESULTADO FINAL
Método: T3
Params: [np.int64(11), np.int64(85)]
Score: 0.2792
Sharpe: 0.2852
Retorno medio: 0.000473
Volatilidad: 0.001660
Trades: 30
Optimizando ventana: 2019-01-14 al 2019-02-08

RESULTADO FINAL
Método: MIDPOINT
Params: [np.int64(19), np.int64(99)]
Score: 0.4292
Sharpe: 0.4332
Retorno medio: 0.000686
Volatilidad: 0.001582
Trades: 20
Optimizando ventana: 2019-01-21 al 2019-02-15

RESULTADO FINAL
Método: TRIMA
Params: [np.int64(14), np.int64(72)]
Score: 0.2127
Sharpe: 0.2183
Retorno medio: 0.000511
Volatilidad: 0.002341
Trades: 28
Optimizando ventana: 2019-01-28 al 2019-02-22

RESULTADO FINAL
Método: EMA
Params: [np.int64(14), np.int64(71)]
Score: 0.2637
Sharpe: 0.2683
Retorno medio: 0.000612
Volatilidad: 0.002280
Trades: 23
Optimizando ventana: 2019-02-04 al 2019-03-01

RESULTADO FINAL
Método: EMA
Params: [np.int64(12), np.int64(92)]
Score: 0.1274
Sharpe: 0.1316
Retorno medio: 0.000244
Volatilidad: 0.001853
Trades: 21
Opti

KeyboardInterrupt: 